[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Quantitative-Research-Methods/blob/main/Projects/project_4_oj/project_4_starter.ipynb)

*This notebook runs on Colab as-is. The badge link above and the `GITHUB_RAW` line in the setup cell already point to this repository, so everything installs and loads automatically.*

# Project 4 — A model the brand manager can read
## Orange-juice brand choice: accuracy against explicability

**Course:** Quantitative Research Methods  
**Instructor:** Prof. Dr. Christoph Weisser, HSBI  
**Source:** James, Witten, Hastie, Tibshirani & Taylor (2023), *An Introduction to Statistical Learning, with Applications in Python*, Springer. Companion code at [statlearning.com](https://www.statlearning.com).

**Goal.** Build the most accurate classifier you can for the choice of Minute Maid over Citrus Hill; build the most *explicable* one you can defend; measure the gap between them in percentage points; and recommend which one the brand manager should deploy.

Read `README.md` in this folder first — it is the brief. This notebook gives you the data, the fixed held-out test set, the baseline, and the scoring helper. Everything after **Step 1** is yours.

## Setup

Run this cell once. The `ISLP` package can be installed with `pip install ISLP`. As an alternative, the same data sets are available as CSVs in the workspace's `ALL CSV FILES - 2nd Edition` folder.


> **Google Colab:** this notebook also runs on Colab out of the box — the setup cell below installs any missing packages and downloads the data automatically.

In [1]:
# --- Setup: runs locally AND on Google Colab --------------------------------
# Silence only the spurious 'encountered in matmul' RuntimeWarnings that the macOS
# Accelerate BLAS emits; real warnings (deprecations, model caveats) stay visible.
import warnings
warnings.filterwarnings('ignore', message='.*encountered in matmul', category=RuntimeWarning)
import importlib.util, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules

def _ensure(pkg, import_name=None):
    """pip-install pkg (quietly) if its import is missing."""
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

if IN_COLAB:  # Colab ships numpy/pandas/sklearn/statsmodels; add course extras
    for _pkg, _imp in [('ISLP', 'ISLP')]:
        _ensure(_pkg, _imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2024)
plt.rcParams['figure.dpi'] = 110

try:
    from ISLP import load_data
    HAVE_ISLP = True
except ImportError:
    HAVE_ISLP = False
    print('ISLP not installed; using CSV / URL fallbacks.')

# Local CSV location (repo layout first, then legacy paths, then a data/ cache).
_CANDIDATES = ['../ALL CSV FILES - 2nd Edition',
               'ALL CSV FILES - 2nd Edition',
               '../../ALL CSV FILES - 2nd Edition', 'data']
CSV = next((p for p in _CANDIDATES if os.path.isdir(p)), 'data')

# GITHUB_RAW lets a fresh Colab runtime fetch any
# CSV that is neither in ISLP nor already local (spaces in the folder -> %20).
GITHUB_RAW = ('https://raw.githubusercontent.com/ChrisW09/Quantitative-Research-Methods/main/'
              'ALL%20CSV%20FILES%20-%202nd%20Edition')

# The four datasets NOT in the ISLP package -> load from the book's official
# site so the notebook works on a fresh Colab even before the repo is published.
KNOWN_URLS = {
    'Advertising': 'https://www.statlearning.com/s/Advertising.csv',
    'Heart':       'https://www.statlearning.com/s/Heart.csv',
    'Income1':     'https://www.statlearning.com/s/Income1.csv',
    'Income2':     'https://www.statlearning.com/s/Income2.csv',
}

def load(name, **read_csv_kwargs):
    """Load a course dataset. Order: ISLP package -> R datasets -> local CSV
    -> official book URL -> your GitHub repo. Works locally and on Colab."""
    if HAVE_ISLP:
        try:
            return load_data(name)
        except Exception:
            pass
    if name == 'USArrests':                       # classic R dataset, not in ISLP
        try:
            import statsmodels.api as sm
            return sm.datasets.get_rdataset('USArrests', 'datasets').data
        except Exception:
            pass
    path = f'{CSV}/{name}.csv'
    if os.path.exists(path):                      # running from the repo (local)
        return pd.read_csv(path, **read_csv_kwargs)
    remotes = ([KNOWN_URLS[name]] if name in KNOWN_URLS else []) + [f'{GITHUB_RAW}/{name}.csv']
    for url in remotes:                           # fresh Colab: stream over https
        try:
            return pd.read_csv(url, **read_csv_kwargs)
        except Exception:
            continue
    raise FileNotFoundError(
        f"Could not load {name!r}. Put the CSV in '{CSV}/' or check your connection for the GITHUB_RAW fallback.")


ISLP not installed; using CSV / URL fallbacks.


## 1. The data — a first look

`OJ` records 1 070 individual orange-juice purchases in five stores: which brand was bought, the two shelf prices, the discounts and in-store specials running that week, and a running measure of the customer's brand loyalty to Citrus Hill.

In [2]:
OJ = load('OJ')

# The ISLP package hands back `Purchase` and `Store7` as pandas Categoricals; the
# CSV fallback hands back plain strings. Pin both to strings so that every table
# below, and every one-hot column name, is the same whichever route the data took.
OJ['Purchase'] = OJ['Purchase'].astype(str)
OJ['Store7'] = OJ['Store7'].astype(str)

print('shape:', OJ.shape)
print()
print(OJ['Purchase'].value_counts().to_frame('purchases')
        .assign(share=lambda d: (d.purchases / len(OJ)).round(3)))
print()
print('MM = Minute Maid (the brand we are asked to explain), CH = Citrus Hill.')

shape: (1070, 18)

          purchases  share
Purchase                  
CH              653   0.61
MM              417   0.39

MM = Minute Maid (the brand we are asked to explain), CH = Citrus Hill.


In [3]:
# The 17 candidate predictors, grouped by what they describe ------------------
GROUPS = {
    'customer': ['LoyalCH'],
    'timing'  : ['WeekofPurchase'],
    'store'   : ['StoreID', 'STORE', 'Store7'],
    'price'   : ['PriceCH', 'PriceMM', 'SalePriceCH', 'SalePriceMM',
                 'PriceDiff', 'ListPriceDiff'],
    'promo'   : ['DiscCH', 'DiscMM', 'PctDiscCH', 'PctDiscMM',
                 'SpecialCH', 'SpecialMM'],
}
for g, cols in GROUPS.items():
    print(f'{g:9s} ({len(cols)}): ' + ', '.join(cols))
print()
print(OJ[['LoyalCH', 'PriceDiff', 'DiscMM', 'SpecialMM']].describe().round(3))

customer  (1): LoyalCH
timing    (1): WeekofPurchase
store     (3): StoreID, STORE, Store7
price     (6): PriceCH, PriceMM, SalePriceCH, SalePriceMM, PriceDiff, ListPriceDiff
promo     (6): DiscCH, DiscMM, PctDiscCH, PctDiscMM, SpecialCH, SpecialMM

        LoyalCH  PriceDiff    DiscMM  SpecialMM
count  1070.000   1070.000  1070.000   1070.000
mean      0.566      0.146     0.123      0.162
std       0.308      0.272     0.214      0.368
min       0.000     -0.670     0.000      0.000
25%       0.325      0.000     0.000      0.000
50%       0.600      0.230     0.000      0.000
75%       0.851      0.320     0.230      0.000
max       1.000      0.640     0.800      1.000


In [4]:
# Two properties of these columns worth knowing before you model them --------
# (a) several price columns are exact arithmetic functions of the others,
print('SalePriceMM  == PriceMM - DiscMM        :',
      np.allclose(OJ.SalePriceMM, OJ.PriceMM - OJ.DiscMM))
print('PriceDiff    == SalePriceMM - SalePriceCH:',
      np.allclose(OJ.PriceDiff, OJ.SalePriceMM - OJ.SalePriceCH))
print('ListPriceDiff== PriceMM - PriceCH        :',
      np.allclose(OJ.ListPriceDiff, OJ.PriceMM - OJ.PriceCH))
# so the design matrix carries far fewer independent columns than it appears to:
num = OJ.drop(columns=['Purchase', 'Store7'])
print(f'\nnumeric predictors: {num.shape[1]}   matrix rank: '
      f'{np.linalg.matrix_rank(num.values)}')
# (b) store identity is recorded three times over:
print()
print(pd.crosstab(OJ.StoreID, [OJ.STORE, OJ.Store7]))

SalePriceMM  == PriceMM - DiscMM        : True
PriceDiff    == SalePriceMM - SalePriceCH: True
ListPriceDiff== PriceMM - PriceCH        : True

numeric predictors: 16   matrix rank: 12

STORE      0    1    2    3    4
Store7   Yes   No   No   No   No
StoreID                         
1          0  157    0    0    0
2          0    0  222    0    0
3          0    0    0  196    0
4          0    0    0    0  139
7        356    0    0    0    0


**What that leaves you to decide.** `PriceDiff` is negative when Minute Maid is the cheaper of the two on the shelf that week. Store identity appears as `StoreID`, again as `STORE`, and again (collapsed to one store versus the rest) as `Store7` — all three are the same nominal label, and none of them is a quantity, although a tree will happily split on `StoreID <= 2.5` as though it were. Whether store identity belongs in a model the brand manager will act on is part of what you must argue in Step 1.

## 2. The held-out test set — fixed here, not to be touched until Step 5

The split below is seeded so that every student in the course reports a number computed on the *same* 268 purchases. Fit, tune and compare on the training set only — by cross-validation or a validation split carved out of the training data (Chapter 5). Touching `X_test` before Step 5 turns your headline number into a training accuracy, and the whole point of the exercise is that it is not one.

In [5]:
from sklearn.model_selection import train_test_split

y = (OJ['Purchase'] == 'MM').astype(int)          # 1 = Minute Maid, 0 = Citrus Hill
X = pd.get_dummies(OJ.drop(columns='Purchase'), drop_first=True).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2024, stratify=y)   # DO NOT change these

print('train:', X_train.shape, ' test:', X_test.shape)
print('MM share  train: %.3f   test: %.3f' % (y_train.mean(), y_test.mean()))
print('predictor columns after one-hot encoding:', list(X.columns))

train: (802, 17)  test: (268, 17)
MM share  train: 0.390   test: 0.388
predictor columns after one-hot encoding: ['WeekofPurchase', 'StoreID', 'PriceCH', 'PriceMM', 'DiscCH', 'DiscMM', 'SpecialCH', 'SpecialMM', 'LoyalCH', 'SalePriceMM', 'SalePriceCH', 'PriceDiff', 'PctDiscMM', 'PctDiscCH', 'ListPriceDiff', 'STORE', 'Store7_Yes']


If your Step 1 argument is that some columns should be dropped, do **not** re-split. Drop them from copies of these frames — `X_train.drop(columns=[...])` — so the rows, and therefore the 268 test purchases, stay identical for everyone.

## 3. The baseline to beat

The manager's current rule of thumb is "most people buy Citrus Hill". A model that cannot beat that rule has no claim on her budget, and saying so clearly is a perfectly good result.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('majority class in the training set:',
      'CH' if y_train.mean() < 0.5 else 'MM')
print('BASELINE test accuracy: %.3f' % accuracy_score(y_test, baseline.predict(X_test)))
print('\n-> it identifies none of the %d Minute Maid buyers in the test set.'
      % int(y_test.sum()))

majority class in the training set: CH
BASELINE test accuracy: 0.612

-> it identifies none of the 104 Minute Maid buyers in the test set.


## 4. The scoring helper — call it once per model, in Step 5

Everyone's numbers must be computed the same way, so use this rather than writing your own. It scores an **already fitted** model on the held-out set, records the row, and reports the two numbers the memo needs: overall accuracy, and the share of the test set's Minute Maid buyers the model actually finds.

In [7]:
RESULTS = []

def evaluate(name, model, X_eval=None):
    """Score a FITTED model on the held-out test set and record one table row.

    name    : label as it should appear in the memo's comparison table
    model    : a fitted classifier
    X_eval  : pass your own test frame if you dropped columns in Step 1
              (must be X_test's rows, in X_test's order); otherwise X_test.
    """
    Xe = X_test if X_eval is None else X_eval
    pred = model.predict(Xe)
    acc = accuracy_score(y_test, pred)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    mm_found = tp / (tp + fn)
    RESULTS.append({'model': name, 'test accuracy': round(acc, 3),
                    'MM buyers found': round(mm_found, 3),
                    'errors': int(fp + fn)})
    print(f'{name:34s} accuracy {acc:.3f}   MM found {mm_found:.3f}   '
          f'({int(fp + fn)} of {len(y_test)} wrong)')
    return acc


def results_table():
    """The comparison table your memo must contain, best model first."""
    return (pd.DataFrame(RESULTS)
              .sort_values('test accuracy', ascending=False)
              .reset_index(drop=True))


evaluate('baseline: always Citrus Hill', baseline)   # the row you must beat
results_table()

baseline: always Citrus Hill       accuracy 0.612   MM found 0.000   (104 of 268 wrong)


,model,test accuracy,MM buyers found,errors
0,baseline: always Citrus Hill,0.612,0.0,104


---

# Your work starts here

Add your code cells under the headings below. Each step names the chapter whose
method applies. Keep the whole notebook runnable top-to-bottom: you will be
handing in this file, and a memo whose numbers cannot be reproduced from it is
worth nothing.

## Step 1 — Decide what goes into the model, and say why

Argue, in a short markdown cell, which of the 17 encoded columns you keep. Two questions have to be answered rather than skipped:

1. **Store identity.** `StoreID`, `STORE` and `Store7_Yes` are three encodings of one nominal label. Keeping it as an integer lets a tree treat store 4 as "twice store 2"; one-hot encoding it treats the stores as the categories they are; dropping it says store identity is not something the brand manager can act on. Pick one, and defend it — the choice is worth measuring both ways.
2. **The redundant price columns.** Only some of §1's arithmetic identities can survive in a model you intend to *interpret*: if `PriceDiff`, `SalePriceMM` and `DiscMM` are all present, an importance ranking splits one effect across three columns.

Whatever you decide, do it by dropping columns from copies of `X_train` / `X_test` — not by re-splitting.

## Step 2 — The most accurate model you can build (Chapter 8)

Fit at least four: a single tree, **bagging**, a **random forest**, and **gradient boosting**. Tune what matters (`ccp_alpha` for the tree; `max_features` for the forest; `n_estimators`, `learning_rate`, `max_depth` for boosting) **by cross-validation inside the training set** (Chapter 5) — `GridSearchCV` as in the Chapter 8 lab. Do not select a model by its test accuracy.

Report the cross-validated accuracy *and* its standard deviation across folds for each candidate. You will need the spread later to say whether the differences you find are real.

## Step 3 — The most interpretable model you can defend (Chapter 8, with Chapter 4)

Now build the model the manager could be shown. A shallow tree is the obvious candidate — fit one, choose its size by cross-validation, and **draw it** with `plot_tree` (or print it with `export_text`) so that every split is legible. A model that needs a paragraph of explanation is not the model this step is asking for.

A tree is not the only explicable model in the course. Chapter 4's classifiers are also models you can read a coefficient off; fitting one costs you two lines and tells you whether the tree is really the best *readable* option available. Report it in the same table.

## Step 4 — Which variables matter, and which of them she can change

Rank the predictors — `feature_importances_` from the ensembles, the split order of your tree, coefficients if you fitted a Chapter 4 model. Then separate the ranking into two lists:

* variables that **describe** the customer, and
* variables the brand manager could **change next quarter**.

The second list is the one the memo is about. For the variables in it, quantify the effect: what happens to the Minute Maid share when the lever moves, and *for whom* — the answer is not the same for every customer, and the interesting part of this project is finding out where the lever works and where the money is wasted.

## Step 5 — The single final evaluation

Now, and only now, score your models on the held-out set with `evaluate(...)`, once each, and print `results_table()`. Include the baseline row, every Chapter 8 model, your interpretable model, and any Chapter 4 comparison.

Then state the gap between your best model and your interpretable one **in percentage points**, and put it in context: it is 268 test purchases, so one percentage point is under three purchases. Is the gap larger than the fold-to-fold spread you measured in Step 2?

## Step 6 — The memo

Write it in a final markdown cell, one page, addressed to the brand manager. It must contain the comparison table, the readable tree, the two variable lists, your recommendation, and the accuracy cost of that recommendation in percentage points. The required numbers are listed in `README.md` under *What you must report*. Caveats belong in the memo, not in a footnote: prices in this data were set by the stores, not assigned by you, so be careful about the verbs you use.